# BGR 01

In [ ]:
pip install "rembg[cpu]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 41.1 MB/s eta 0:00:00


In [ ]:
from rembg import remove
from PIL import Image
import os

# In Colab, tkinter's filedialog cannot be used directly as there's no display.
# Instead, define input and output folders manually, or use google.colab.drive for Google Drive integration.
# Example:
input_folder = "/content/input" # Create this folder and upload your images here
output_folder = "/content/output" # The processed images will be saved here

# Create the folders if they don't exist
os.makedirs(input_folder, exist_ok=True)
os.makedirs(output_folder, exist_ok=True)

# Check if folders selected (now automatically defined)
if not input_folder or not output_folder:
    print("Folder paths are not defined properly!")
    exit()

for filename in os.listdir(input_folder):
    if filename.lower().endswith((".jpg", ".png", ".jpeg", ".gif", ".bmp", ".tiff")):
        input_path = os.path.join(input_folder, filename)
        output_path = os.path.join(
            output_folder, os.path.splitext(filename)[0] + ".png"
        )

        try:
            with open(input_path, "rb") as f:
                result = remove(f.read())

            with open(output_path, "wb") as f:
                f.write(result)
            print(f"Processed: {filename}")
        except Exception as e:
            print(f"Error processing {filename}: {e}")

print("Done!")

  0%|                                               | 0.00/176M [00:00<?, ?B/s]

Processed: IMG_4137.JPG
Processed: IMG_4144.JPG
Processed: IMG_4142.JPG
Processed: IMG_4139.JPG
Processed: IMG_4143.JPG
Processed: IMG_4145.JPG
Processed: IMG_4140.JPG
Processed: IMG_4141.JPG
Done!


# BGR 02 - Hugging Face

## install libraries

In [ ]:
!pip install transformers torchvision pillow huggingface_hub kornia

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 80.6 MB/s eta 0:00:00


## login with token

In [ ]:
from huggingface_hub import login
login("hf_token")

## mount google drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## BGR

In [ ]:
from PIL import Image
import torch
from torchvision import transforms
from transformers import AutoModelForImageSegmentation
import os

# 🔹 SET YOUR FOLDERS HERE
input_folder = "/content/drive/MyDrive/input"
output_folder = "/content/drive/MyDrive/output"

os.makedirs(output_folder, exist_ok=True)

# 🔹 Device setup
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Using device:", device)

# 🔹 Load model (with auth)
model = AutoModelForImageSegmentation.from_pretrained(
    "briaai/RMBG-2.0",
    trust_remote_code=True,
    token=True
).to(device).eval()

# 🔥 Speed boost if GPU
if device == 'cuda':
    model = model.half()

# 🔹 Image transform
transform_image = transforms.Compose([
    transforms.Resize((1024, 1024)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

def remove_bg(input_path, output_path):
    image = Image.open(input_path).convert("RGB")

    input_tensor = transform_image(image).unsqueeze(0).to(device)

    if device == 'cuda':
        input_tensor = input_tensor.half()

    with torch.no_grad():
        preds = model(input_tensor)[-1].sigmoid().cpu()

    pred = preds[0].squeeze()
    mask = transforms.ToPILImage()(pred).resize(image.size)

    image.putalpha(mask)
    image.save(output_path)

# 🔹 Process all images
for filename in os.listdir(input_folder):
    if filename.lower().endswith((".jpg", ".png", ".jpeg")):
        input_path = os.path.join(input_folder, filename)
        output_path = os.path.join(
            output_folder,
            os.path.splitext(filename)[0] + ".png"
        )

        print("Processing:", filename)
        remove_bg(input_path, output_path)

print("✅ Done!")

Using device: cuda


config.json:   0%|          | 0.00/405 [00:00<?, ?B/s]

BiRefNet_config.py:   0%|          | 0.00/298 [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/briaai/RMBG-2.0:
- BiRefNet_config.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


birefnet.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/briaai/RMBG-2.0:
- birefnet.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.12/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)


model.safetensors:   0%|          | 0.00/885M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/754 [00:00<?, ?it/s]

Processing: Copy of IMG_4144.JPG
Processing: Copy of IMG_4142.JPG
Processing: Copy of IMG_4141.JPG
Processing: Copy of IMG_4146.JPG
Processing: Copy of IMG_4140.JPG
Processing: Copy of IMG_4138.JPG
Processing: Copy of IMG_4145.JPG
Processing: Copy of IMG_4143.JPG
Processing: Copy of IMG_4139.JPG
Processing: Copy of IMG_4137.JPG
Processing: Copy of IMG_4153.JPG
Processing: Copy of IMG_4149.JPG
Processing: Copy of IMG_4151.JPG
Processing: Copy of IMG_4150.JPG
Processing: Copy of IMG_4152.JPG
Processing: Copy of IMG_4154.JPG
Processing: Copy of IMG_4155.JPG
Processing: Copy of IMG_4156.JPG
Processing: Copy of IMG_4147.JPG
Processing: Copy of IMG_4148.JPG
Processing: Copy of IMG_4162.JPG
Processing: Copy of IMG_4161.JPG
Processing: Copy of IMG_4160.JPG
Processing: Copy of IMG_4166.JPG
Processing: Copy of IMG_4159.JPG
Processing: Copy of IMG_4164.JPG
Processing: Copy of IMG_4158.JPG
Processing: Copy of IMG_4165.JPG
Processing: Copy of IMG_4163.JPG
Processing: Copy of IMG_4157.JPG
Processing